## Start modelering van voorspellingsmodel

### Import data

In [1038]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from sklearn import metrics

df = pd.read_csv('../../csv/new_train_data/train_CarBreakDown.csv')

In [ ]:
used_features =['vehicle_brand','vehicle_age_years','mileage_km','engine_hours','last_service_km_ago','oil_quality_pct','avg_trip_length_km','weather_exposure','fuel_type','tyre_type']
x = df[used_features]
y = df[['breakdown_next_30_days']]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [1040]:
x

,vehicle_brand,vehicle_age_years,mileage_km,engine_hours,last_service_km_ago,oil_quality_pct,avg_trip_length_km,weather_exposure,fuel_type,cleanliness_score,driver_satisfaction_score,tyre_type
0,Toyota,5.0,92828.113285,2986.923461,1845.744691,54.060416,6.464345,high,diesel,72.246829,4.719607,winter
1,Volvo,11.0,66541.746883,1436.255252,5158.229238,52.388355,29.987997,medium,diesel,34.610147,4.221209,summer
2,Toyota,19.0,120040.213061,3799.069247,33057.522511,27.637197,186.883521,medium,diesel,85.933503,6.185589,summer
3,Volkswagen,4.0,58441.648247,1956.250851,4080.782658,52.013925,140.844507,medium,petrol,88.876837,8.334875,summer
4,Ford,2.0,113057.424374,3818.890986,2585.704820,48.387331,21.020533,low,diesel,86.677049,6.622987,summer
...,...,...,...,...,...,...,...,...,...,...,...,...
912,Renault,6.0,131602.464358,4586.327861,32185.769120,83.247676,6.448159,low,electric,94.249149,6.142659,all-season
913,Toyota,9.0,127224.810750,4106.036923,23433.319506,69.923976,1.773332,medium,petrol,62.945838,5.346505,summer
914,Ford,19.0,74204.367905,2323.811128,6490.998514,60.864621,39.444093,low,diesel,88.354147,5.944547,winter
915,BMW,10.0,198492.993706,6643.176908,3564.136473,82.555275,9.060255,low,diesel,64.073009,6.061162,all-season


In [1041]:
y

,breakdown_next_30_days
0,0
1,0
2,0
3,0
4,0
...,...
912,0
913,1
914,0
915,0


### Encoding van categorische velden

In [1042]:
# !pip install --upgrade category_encoders
import category_encoders as ce

category_encoder = ce.OrdinalEncoder(cols = ['vehicle_brand','weather_exposure','fuel_type','tyre_type'])
x_encoded = category_encoder.fit_transform(x_train)
x_test_encoded = category_encoder.transform(x_test)
x_encoded


,vehicle_brand,vehicle_age_years,mileage_km,engine_hours,last_service_km_ago,oil_quality_pct,avg_trip_length_km,weather_exposure,fuel_type,cleanliness_score,driver_satisfaction_score,tyre_type
95,1,4.0,50713.080246,1909.258457,521.948846,61.645665,4.429587,1,1,60.422880,4.532415,1
708,2,11.0,132396.000000,3695.825175,17285.940878,10.876016,37.831165,2,2,35.436728,5.203857,2
509,3,6.0,83890.420100,3132.325509,16080.851770,29.194847,9.058284,1,3,71.423516,6.018523,1
611,2,10.0,108900.842941,3925.339078,15646.862613,31.203292,4.814017,2,4,58.395397,8.723527,2
379,1,16.0,192576.000000,7126.876676,2740.365129,16.863613,151.444730,3,1,87.311928,6.520118,1
...,...,...,...,...,...,...,...,...,...,...,...,...
2,5,19.0,120040.213061,3799.069247,33057.522511,27.637197,186.883521,1,4,85.933503,6.185589,2
585,6,11.0,78331.650796,2800.405487,10214.639709,56.288161,27.963553,3,2,45.576631,6.174282,3
426,5,9.0,153309.206086,5375.663134,24911.692733,59.490156,4.870460,2,1,57.163757,8.510578,2
911,7,6.0,73047.389818,2678.999483,14085.805874,72.835609,43.227761,3,4,73.314438,9.001056,2


### Train model via Random forest

In [1043]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(criterion='entropy',n_estimators=100, max_samples = 200)
clf = clf.fit(x_encoded, y_train.values.ravel()) # values.ravel() because a warning appears about using a 1d array so we need to flatten it.

In [1044]:
clf_prediction = clf.predict(x_test_encoded)
print(clf_prediction)

[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]


In [1045]:
print(f"Accuracy: {metrics.accuracy_score(y_test, clf_prediction)}")

Accuracy: 0.782608695652174


In [1046]:
confusion_matrix(y_test, clf_prediction)

array([[144,   0],
       [ 40,   0]])